# Billups layer debugging

Use this notebook to inspect Bronze, Silver, and Gold with PySpark. Run `billups.pipeline` or the required individual stages before querying a layer. Keep outputs cleared before committing.

In [1]:
import json
import sys
from pathlib import Path

project_root = Path.cwd()
if not (project_root / "billups").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

from pyspark.sql import functions as F
from billups.common import build_spark

spark = build_spark("billups-data-debugging")

26/09/16 09:08:31 WARN Utils: Your hostname, Andres-Laptop.local resolves to a loopback address: 127.0.0.1; using 192.168.0.9 instead (on interface en0)
26/09/16 09:08:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/16 09:08:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
bronze_dir = project_root / "data" / "bronze"
silver_dir = project_root / "data" / "silver"
gold_dir = project_root / "data" / "gold"

print("Bronze:", bronze_dir.exists())
print("Silver:", silver_dir.exists())
print("Gold:", gold_dir.exists())

Bronze: True
Silver: True
Gold: True


## Bronze

Bronze preserves the raw source grain in Parquet. Actions below are deliberately bounded.

In [4]:
bronze_transactions = spark.read.parquet(str(bronze_dir / "historical_transactions"))
bronze_merchants = spark.read.parquet(str(bronze_dir / "merchants"))
bronze_transactions.printSchema()
bronze_transactions.limit(5).show(truncate=False)
bronze_merchants.select("merchant_id", "merchant_name").limit(5).show(truncate=False)

root
 |-- authorized_flag: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city_id: long (nullable = true)
 |-- installments: long (nullable = true)
 |-- category: string (nullable = true)
 |-- merchant_category_id: long (nullable = true)
 |-- merchant_id: string (nullable = true)
 |-- month_lag: long (nullable = true)
 |-- purchase_date: string (nullable = true)
 |-- state_id: long (nullable = true)
 |-- subsector_id: long (nullable = true)
 |-- purchase_amount: double (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = true)

+---------------+---------------+-------+------------+--------+--------------------+---------------+---------+-------------------+--------+------------+---------------+-------------------------------+--------------------------+
|authorized_flag|customer_id    |city_id|installments|category|merchant_category_id|merchant_id    |month_lag|purchase_date      |state_id|subsector_

## Silver

Silver contains validated amounts and timestamps, normalized categories, and merchant-name enrichment.

In [6]:
silver_transactions = spark.read.parquet(str(silver_dir / "transactions"))
silver_transactions.printSchema()
silver_transactions.select(
    "purchase_date", "purchase_amount", "merchant_id", "merchant_name", "category"
).limit(10).show(truncate=False)

with (silver_dir / "dq.json").open(encoding="utf-8") as stream:
    silver_dq = json.load(stream)
silver_dq

root
 |-- merchant_id: string (nullable = true)
 |-- authorized_flag: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- city_id: long (nullable = true)
 |-- installments: long (nullable = true)
 |-- category: string (nullable = true)
 |-- merchant_category_id: long (nullable = true)
 |-- month_lag: long (nullable = true)
 |-- state_id: long (nullable = true)
 |-- subsector_id: long (nullable = true)
 |-- source_file_name: string (nullable = true)
 |-- bronze_load_timestamp: timestamp (nullable = true)
 |-- purchase_date: timestamp (nullable = true)
 |-- purchase_amount: decimal(28,2) (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_name_ambiguous: boolean (nullable = true)
 |-- merchant_lookup_matched: boolean (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+-------------------+---------------+---------------+--------------------+--------+
|purchase_date      |purchase_amount|merchant_id    |merchant_name    

{'ambiguous_merchant_ids': 41,
 'denied': 628174,
 'missing_merchant_id': 34570,
 'nonpositive_amount': 0,
 'row_count': 7274367,
 'total_amount': '146228071619.26',
 'unknown_authorization': 0,
 'unknown_category': 44625,
 'unknown_city': 0,
 'unknown_installments': 50,
 'unknown_state': 661973,
 'unmatched_merchant_lookup': 34570}

In [7]:
silver_transactions.groupBy("authorized_flag").agg(
    F.count("*").alias("attempt_count"),
    F.sum("purchase_amount").alias("total_amount"),
).orderBy("authorized_flag").show()

+---------------+-------------+---------------+
|authorized_flag|attempt_count|   total_amount|
+---------------+-------------+---------------+
|              N|       628174| 12636444221.95|
|              Y|      6646193|133591627397.31|
+---------------+-------------+---------------+



In [8]:
silver_merchant_lookup = spark.read.parquet(str(silver_dir / "merchant_lookup"))
silver_merchant_lookup.printSchema()
silver_merchant_lookup.limit(10).show(truncate=False)

root
 |-- merchant_id: string (nullable = true)
 |-- merchant_name: string (nullable = true)
 |-- merchant_name_ambiguous: boolean (nullable = true)
 |-- merchant_lookup_matched: boolean (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+---------------+---------------------+-----------------------+-----------------------+--------------------------+
|merchant_id    |merchant_name        |merchant_name_ambiguous|merchant_lookup_matched|silver_load_timestamp     |
+---------------+---------------------+-----------------------+-----------------------+--------------------------+
|M_ID_000087311e|Benjamin Puga inc    |false                  |true                   |2026-09-16 11:53:55.842895|
|M_ID_0000ab0b2d|Ericka Huff inc      |false                  |true                   |2026-09-16 11:53:55.842895|
|M_ID_0000fd7caf|Donald Gartner inc   |false                  |true                   |2026-09-16 11:53:55.842895|
|M_ID_0001c38687|Vincent Mabon inc    |false    

In [9]:
silver_merchant_conflicts = spark.read.parquet(str(silver_dir / "merchant_conflicts"))
silver_merchant_conflicts.printSchema()
silver_merchant_conflicts.limit(10).show(truncate=False)

root
 |-- merchant_id: string (nullable = true)
 |-- nonblank_names: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- distinct_name_count: integer (nullable = true)
 |-- silver_load_timestamp: timestamp (nullable = true)

+---------------+---------------------------------------------------------------------------------------+-------------------+--------------------------+
|merchant_id    |nonblank_names                                                                         |distinct_name_count|silver_load_timestamp     |
+---------------+---------------------------------------------------------------------------------------+-------------------+--------------------------+
|M_ID_0039220eb3|[Clark Corder inc, Jeffrey Estrada inc]                                                |2                  |2026-09-16 11:53:55.842895|
|M_ID_00a6ca8a8a|[John Miller 7 inc, Nilda Richter inc]                                                 |2                  |2026-09-16 11

26/09/16 12:12:17 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 209950 ms exceeds timeout 120000 ms
26/09/16 12:12:17 WARN SparkContext: Killing executors is not supported by current scheduler.
26/09/16 12:12:17 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:80)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:642)
	at org.apache.spark.executor.Executor.reportHeartBeat(Executor.scala:1240)
	at o

## Gold

Gold tables contain the business answers consumed by the report and dashboard.

In [5]:
gold_tables = sorted(path.name for path in gold_dir.iterdir() if path.is_dir())
gold_tables


['data_quality_merchant_conflicts',
 'data_quality_summary',
 'data_quality_warning_samples',
 'q1_top_merchants',
 'q2_merchant_state',
 'q3_category_hours',
 'q4_city_category',
 'q4_popular_merchants',
 'q5_categories',
 'q5_cities',
 'q5_hours',
 'q5_installments',
 'q5_months']

In [9]:
q1 = spark.read.parquet(str(gold_dir / "q1_top_merchants"))
q1.filter((F.col("year_month") == "2017-01") & (F.col("city_id") == 2)).orderBy("rank").show(5, truncate=False)

+----+----------+-------+---------------+---------------------+-------------+-------------+
|rank|year_month|city_id|merchant_id    |merchant_name        |total_amount |attempt_count|
+----+----------+-------+---------------+---------------------+-------------+-------------+
|1   |2017-01   |2      |M_ID_349f038eea|Mark Cipolla inc     |396768.340000|19           |
|2   |2017-01   |2      |M_ID_3da6b9ff1b|Shana Platt inc      |271193.750000|14           |
|3   |2017-01   |2      |M_ID_2efaadc5ca|Ernesto Matthews inc |218559.820000|11           |
|4   |2017-01   |2      |M_ID_20a53d9145|James Luhman inc     |207420.850000|11           |
|5   |2017-01   |2      |M_ID_8510d3fa74|Salvador Adderley inc|203993.930000|9            |
+----+----------+-------+---------------+---------------------+-------------+-------------+



In [10]:
q5_months = spark.read.parquet(str(gold_dir / "q5_months"))
q5_months.orderBy(F.desc("approved_amount_per_observed_day")).show(truncate=False)

+----------+------------------+-----------------+------------------+--------------+-------------------+------------------+-------------+--------------------------------+
|year_month|all_attempt_amount|all_attempt_count|approved_amount   |approved_count|first_observed_date|last_observed_date|observed_days|approved_amount_per_observed_day|
+----------+------------------+-----------------+------------------+--------------+-------------------+------------------+-------------+--------------------------------+
|2017-12   |17130706682.780000|852200           |15654272191.850000|778747        |2017-12-01         |2017-12-31        |31           |504976522.317742                |
|2017-11   |14190873326.760000|706032           |13014753496.240000|647493        |2017-11-01         |2017-11-30        |30           |433825116.541333                |
|2018-01   |14340910058.930000|713701           |13212571549.210000|657591        |2018-01-01         |2018-01-31        |31           |426211985.4583

## Stop Spark

Release the local Spark resources when debugging is complete.

In [11]:
spark.stop()